In [25]:
import numpy as np
import gymnasium as gym

env = gym.make("Taxi-v4")
n_states = env.observation_space.n
n_actions = env.action_space.n
print("Taxi states:", n_states, "Actions:", n_actions)

Taxi states: 500 Actions: 6


In [26]:
def q_from_v(env, V, s, gamma):
    """Q(s,a) cho mọi a bằng one-step lookahead.
    done=True: không cộng giá trị tương lai vì episode đã kết thúc."""
    P = env.unwrapped.P  # P[s][a] = [(prob, next_state, reward, done), ...]
    q = np.zeros(env.action_space.n)
    for a in range(env.action_space.n):
        for prob, s_next, r, done in P[s][a]:
            q[a] += prob * (r + gamma * V[s_next] * (not done))
    return q

In [27]:
def value_iteration(env, gamma=0.9, theta=1e-6):
    n_states = env.observation_space.n
    V = np.zeros(n_states)
    while True:
        delta = 0.0
        for s in range(n_states):
            v_old = V[s]
            # Bellman optimality: lấy max theo hành động
            V[s] = np.max(q_from_v(env, V, s, gamma))
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:  # gần điểm bất động V*
            break
    # trích policy greedy từ V* (làm 1 lần ở cuối)
    policy = np.array([np.argmax(q_from_v(env, V, s, gamma)) for s in range(n_states)])
    return V, policy


In [28]:
def policy_iteration(env, gamma=0.9, theta=1e-6):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    P = env.unwrapped.P

    policy = np.random.choice(n_actions, n_states)  # 1. policy ngẫu nhiên
    V = np.zeros(n_states)
    stable = False

    while not stable:
        # 2. Policy Evaluation (đánh giá policy hiện tại)
        while True:
            delta = 0.0
            for s in range(n_states):
                v_old = V[s]
                a = policy[s]
                V[s] = sum(prob * (r + gamma * V[s_next] * (not done))
                           for prob, s_next, r, done in P[s][a])
                delta = max(delta, abs(v_old - V[s]))
            if delta < theta:
                break

        # 3. Policy Improvement (cải tiến policy)
        stable = True
        for s in range(n_states):
            old_action = policy[s]
            q = q_from_v(env, V, s, gamma)
            best = np.argmax(q)
            # chỉ đổi khi thực sự tốt hơn (tránh dao động do hòa / sai số số thực)
            if q[best] > q[old_action] + 1e-10:
                policy[s] = best
                stable = False

    return V, policy

In [29]:
np.random.seed(0)
V_vi, policy_vi = value_iteration(env)
V_pi, policy_pi = policy_iteration(env)
print("Mean Value (VI):", np.mean(V_vi))
print("Mean Value (PI):", np.mean(V_pi))
print("Different actions:", np.sum(policy_vi != policy_pi))
print("Max |V_vi - V_pi|:", np.max(np.abs(V_vi - V_pi)))

Mean Value (VI): 2.4679209766162074
Mean Value (PI): 2.4679209766162074
Different actions: 65
Max |V_vi - V_pi|: 0.0
